# Phase 4 — Scorecard (PD Ins + PD Css)

Train accepted-only application PD scorecards on `abt_app` + `decisions`.

Functions are defined in this notebook first; they are ported to `src/credit_scoring/scorecard/` for Kedro later.


In [1]:
# Import libraries
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", None)


In [2]:
# Phase 4 parameters — keep in sync with conf/base/parameters.yml (scorecard section)
SCORECARD_PARAMS = {
    "target": "default12",
    "time_col": "period",
    "id_col": "aid",
    "train_end_period": "198412",
    "valid_start_period": "198501",
    "ncategories_int": 4,
    "minimum_share_int": 0.03,
    "symbol_missing": "Missing",
    "symbol_other": "<OTHERS>",
    "ncategories_nom": 4,
    "iv_min": 0.02,
    "gini_min": 0.05,
    "psi_max": 0.25,
    "ar_diff_max": 0.20,
    "pvalue_max": 0.05,
    "vif_max": 5.0,
    "pearson_max": 0.7,
    "ar_diff_model_max": 0.10,
    "factor": 20 / np.log(2),
    "offset": 600 - (20 / np.log(2)) * np.log(50),
    "woe_epsilon": 1e-4,
    "tree_random_state": 1234,
    "max_features": 12,
    "epsilon": 1e-4,
}


### Parameters — screening and model thresholds

| Key | Role |
|-----|------|
| `iv_min` / `gini_min` / `psi_max` / `ar_diff_max` | Feature prescreen gates |
| `pvalue_max` / `vif_max` / `pearson_max` / `ar_diff_model_max` | Forward-selection model gates |
| `factor` / `offset` | PDO scaling (points table) |


## Functions

Defined here first; port to `src/credit_scoring/scorecard/` when wiring Kedro pipelines.

In [3]:
# --- Partition + load / target prep ---

LEAKAGE_COLS = [
    "default3", "default6", "default12", "decision", "decline_reason",
    "act3_n_arrears", "act3_n_arrears_days", "act3_n_good_days",
    "act6_n_arrears", "act6_n_arrears_days", "act6_n_good_days",
    "act9_n_arrears", "act9_n_arrears_days", "act9_n_good_days",
    "act12_n_arrears", "act12_n_arrears_days", "act12_n_good_days",
    "act_cus_active",
]
ID_COLS = ["cid", "aid", "period", "product"]


def partition_abt(df, train_end, valid_start, time_col="period"):
  """Split ABT into non-overlapping train and validation windows."""
  work = df.copy()
  df_train = work[work[time_col] <= train_end]
  df_valid = work[work[time_col] >= valid_start]
  return df_train, df_valid


def prepare_target(df, target="default12"):
  """Map raw default labels to binary {0, 1} and drop missing targets."""
  out = df.copy()
  out[target] = out[target].map({".i": 0, ".d": 0, 0: 0, 1: 1})
  out = out.dropna(subset=[target])
  out[target] = out[target].astype(int)
  return out


def load_and_prepare_abt(abt_path, decisions_path):
  """Merge application base table with decisions and prepare the target."""
  abt = pd.read_parquet(abt_path)
  decisions = pd.read_parquet(decisions_path)
  merged = abt.merge(
    decisions[["aid", "decision", "decline_reason"]],
    on="aid",
    how="left",
  )
  return prepare_target(merged)

In [ ]:
# Checkpoint 1–2: load + prepare target
df = load_and_prepare_abt("../data/04_feature/abt_app.parquet", "../data/04_feature/decisions.parquet")
assert "decision" in df.columns
assert "default12" in df.columns
assert df["aid"].is_unique
assert set(df["default12"].unique()) <= {0, 1}
print(df.groupby(["product", "decision"])["default12"].mean())
print("rows kept:", len(df), "bad rate:", f"{df['default12'].mean():.2%}")

product  decision
css      A           0.662746
         N           0.740232
ins      A           0.137648
Name: default12, dtype: float64
rows kept: 44602 bad rate: 41.04%


In [4]:
# --- Binning ---

def _bin_params(params):
  return {
    "other_label": params.get("symbol_other", params.get("other_label", "<OTHERS>")),
    "missing_label": params.get("symbol_missing", params.get("missing_label", "Missing")),
    "max_bins": params.get("ncategories_int", params.get("max_bins", 5)),
    "min_bin_size": params.get("minimum_share_int", params.get("min_bin_size", 0.05)),
    "tree_random_state": params.get("tree_random_state", 1234),
    "rare_threshold": params.get("rare_threshold", 0.02),
    "max_groups": params.get("ncategories_nom", 4),
    "nominal_int_threshold": params.get("nominal_int_threshold", 10),
  }


def fit_bin_numeric(train, feature, target, params):
  """Fit decision-tree bins for one numeric feature."""
  cfg = _bin_params(params)
  has_missing = train[feature].isna().any()
  clean = train.dropna(subset=[feature, target]).copy()
  x = clean[[feature]].values
  y = clean[target].values

  tree = DecisionTreeClassifier(
    max_leaf_nodes=cfg["max_bins"],
    min_samples_leaf=max(1, int(cfg["min_bin_size"] * len(clean))),
    random_state=cfg["tree_random_state"],
  )
  tree.fit(x, y)

  thresholds = np.unique(tree.tree_.threshold[tree.tree_.threshold != -2])
  edges = [-np.inf] + list(np.sort(thresholds)) + [np.inf]

  return {
    "type": "numeric",
    "feature": feature,
    "edges": edges,
    "missing_bin": bool(has_missing),
    "missing_label": cfg["missing_label"],
  }


def fit_bin_nominal(train, feature, target, params):
  """Fit risk-ordered nominal groups with rare-category pooling."""
  cfg = _bin_params(params)
  other_label = cfg["other_label"]
  missing_label = cfg["missing_label"]
  rare_threshold = cfg["rare_threshold"]
  max_groups = cfg["max_groups"]

  clean = train.dropna(subset=[feature, target]).copy()
  clean["_cat_norm"] = clean[feature].astype(str)

  freq = clean["_cat_norm"].value_counts(normalize=True)
  rare_cats = set(freq[freq < rare_threshold].index)
  valid_cats = set(freq[freq >= rare_threshold].index)

  stats = (
    clean[clean["_cat_norm"].isin(valid_cats)]
    .groupby("_cat_norm")[target]
    .agg(["mean", "count"])
    .rename(columns={"mean": "event_rate", "count": "n"})
    .sort_values(by="event_rate", ascending=True)
  )

  groups = [[cat] for cat in stats.index]

  while len(groups) > max_groups:
    best_i, best_n = 0, float("inf")
    for i in range(len(groups) - 1):
      combined = sum(stats.loc[c, "n"] for c in groups[i] + groups[i + 1])
      if combined < best_n:
        best_n, best_i = combined, i
    groups[best_i] += groups[best_i + 1]
    groups.pop(best_i + 1)

  category_map = {}
  for idx, grp in enumerate(groups):
    label = f"G{idx + 1:02d}"
    for cat in grp:
      category_map[cat] = label

  for cat in rare_cats:
    category_map[str(cat)] = other_label

  return {
    "type": "nominal",
    "feature": feature,
    "category_map": category_map,
    "other_label": other_label,
    "missing_label": missing_label,
  }


def fit_binning_maps(train, features, target, params):
  """Dispatch numeric or nominal bin fitters for every candidate feature."""
  cfg = _bin_params(params)
  nominal_int_threshold = cfg["nominal_int_threshold"]
  binning_maps = {}

  for feat in features:
    col = train[feat]
    is_nominal = (
      pd.api.types.is_bool_dtype(col)
      or pd.api.types.is_object_dtype(col)
      or pd.api.types.is_string_dtype(col)
      or isinstance(col.dtype, pd.CategoricalDtype)
      or (
        pd.api.types.is_integer_dtype(col)
        and col.nunique(dropna=True) <= nominal_int_threshold
      )
    )
    if is_nominal:
      binning_maps[feat] = fit_bin_nominal(train, feat, target, params)
    else:
      binning_maps[feat] = fit_bin_numeric(train, feat, target, params)

  return binning_maps


def apply_bins(df, binning_maps):
  """Apply fitted bin definitions and add ``{feature}_GRP`` columns."""
  base = df.drop(columns=[c for c in df.columns if c.endswith("_GRP")], errors="ignore")
  new_cols = {}

  for feat, spec in binning_maps.items():
    grp_col = f"{feat}_GRP"

    if spec["type"] == "numeric":
      missing_label = spec.get("missing_label", "Missing")
      edges = spec["edges"]
      cut = pd.cut(base[feat], bins=edges, include_lowest=True, right=True)
      new_cols[grp_col] = cut.astype("string").fillna(missing_label)

    elif spec["type"] == "nominal":
      cat_map = spec["category_map"]
      other_label = spec["other_label"]
      missing_label = spec["missing_label"]

      s = base[feat]
      missing_mask = s.isna()
      norm = s.astype("string")
      mapped = norm.map(cat_map).fillna(other_label)
      mapped.loc[missing_mask] = missing_label
      new_cols[grp_col] = mapped.astype("string")

    else:
      raise ValueError(f"Unknown binning type for {feat}: {spec['type']}")

  grp_df = pd.DataFrame(new_cols, index=base.index)
  return pd.concat([base, grp_df], axis=1).copy()

In [5]:
# --- WOE / IV ---

def build_woe_table(df_binned, feature_grp, target, epsilon):
  """Compute goods, bads, WOE, and IV contribution for one grouped feature."""
  total_good = (df_binned[target] == 0).sum()
  total_bad = (df_binned[target] == 1).sum()

  grouped = (
    df_binned.groupby(feature_grp, observed=False)[target]
    .agg(n="count", bads="sum")
    .reset_index()
    .rename(columns={feature_grp: "bin"})
  )

  grouped["goods"] = grouped["n"] - grouped["bads"]
  grouped["dist_good"] = (grouped["goods"] + epsilon) / (total_good + epsilon)
  grouped["dist_bad"] = (grouped["bads"] + epsilon) / (total_bad + epsilon)
  grouped["woe"] = np.log(grouped["dist_good"] / grouped["dist_bad"])
  grouped["iv_component"] = (grouped["dist_good"] - grouped["dist_bad"]) * grouped["woe"]
  return grouped


def build_woe_maps(df_binned, grp_cols, target, epsilon):
  return {grp: build_woe_table(df_binned, grp, target, epsilon) for grp in grp_cols}


def compute_iv(woe_table):
  return float(max(woe_table["iv_component"].sum(), 0.0))


def build_iv_table(woe_maps):
  rows = []
  for grp, table in woe_maps.items():
    feature = grp[: -len("_GRP")]
    rows.append({"feature": feature, "iv": compute_iv(table)})
  return pd.DataFrame(rows).sort_values("iv", ascending=False).reset_index(drop=True)


def encode_woe(df_binned, woe_maps):
  """Map grouped bins to numeric WOE columns."""
  base = df_binned.drop(columns=[c for c in df_binned.columns if c.endswith("_WOE")], errors="ignore")
  new_cols = {}

  for grp_col, woe_table in woe_maps.items():
    feature = grp_col[: -len("_GRP")]
    woe_col = f"{feature}_WOE"
    bin_to_woe = dict(zip(woe_table["bin"], woe_table["woe"]))
    new_cols[woe_col] = base[grp_col].map(bin_to_woe).fillna(0.0)

  woe_df = pd.DataFrame(new_cols, index=base.index)
  return pd.concat([base, woe_df], axis=1).copy()

In [6]:
# --- Feature screening + model selection ---

def get_candidate_features(df, product):
  """List accepted-only model candidates for one product."""
  work = df.copy()
  work = work[work["product"] == product]
  work = work[work["decision"] == "A"]
  work = work.dropna(subset=["decision", "decline_reason"])

  drop_cols = [c for c in LEAKAGE_COLS + ID_COLS if c in work.columns]
  work = work.drop(columns=drop_cols)

  numeric = [
    c
    for c in work.columns
    if pd.api.types.is_numeric_dtype(work[c]) and not pd.api.types.is_bool_dtype(work[c])
  ]
  nominal = [
    c
    for c in work.columns
    if c not in numeric
    and (
      pd.api.types.is_object_dtype(work[c])
      or pd.api.types.is_string_dtype(work[c])
      or isinstance(work[c].dtype, pd.CategoricalDtype)
      or pd.api.types.is_bool_dtype(work[c])
    )
  ]
  return {"numeric": numeric, "nominal": nominal, "all": numeric + nominal}


def compute_gini(y_true, y_score):
  auc = roc_auc_score(y_true, y_score)
  auc = max(auc, 1 - auc)
  return float(np.clip(2 * auc - 1, 0.0, 1.0))


def compute_psi(train_series, valid_series, epsilon):
  train_dist = train_series.value_counts(normalize=True)
  valid_dist = valid_series.value_counts(normalize=True)

  all_bins = train_dist.index.union(valid_dist.index)
  train_pct = train_dist.reindex(all_bins, fill_value=0) + epsilon
  valid_pct = valid_dist.reindex(all_bins, fill_value=0) + epsilon

  psi_components = (train_pct - valid_pct) * np.log(train_pct / valid_pct)
  return float(max(psi_components.sum(), 0.0))


def check_vif(x):
  x_num = x.select_dtypes(include=[np.number]).copy()
  x_ = x_num.copy()
  x_.insert(0, "_intercept", 1.0)

  vif_values = {}
  for i, col in enumerate(x_.columns):
    if col == "_intercept":
      continue
    vif_values[col] = variance_inflation_factor(x_.values, i)

  return pd.Series(vif_values, name="vif")


def prescreen_features(train_woe, valid_woe, iv_table, params):
  target = params["target"]
  iv_min = params["iv_min"]
  gini_min = params["gini_min"]
  psi_max = params["psi_max"]
  ar_diff_max = params["ar_diff_max"]
  epsilon = params["woe_epsilon"]

  rows = []
  for _, row in iv_table.iterrows():
    feature = row["feature"]
    iv = row["iv"]
    woe_col = f"{feature}_WOE"
    grp_col = f"{feature}_GRP"
    reasons = []

    if iv < iv_min:
      reasons.append(f"IV < {iv_min}")

    gini_train = gini_valid = ar_diff = np.nan

    if woe_col in train_woe.columns:
      gini_train = compute_gini(train_woe[target], train_woe[woe_col])
    else:
      reasons.append("No WOE column (train)")

    if woe_col in valid_woe.columns:
      gini_valid = compute_gini(valid_woe[target], valid_woe[woe_col])
    else:
      reasons.append("No WOE column (valid)")

    if not np.isnan(gini_train) and gini_train < gini_min:
      reasons.append(f"Gini train < {gini_min}")
    if not np.isnan(gini_valid) and gini_valid < gini_min:
      reasons.append(f"Gini valid < {gini_min}")

    if not np.isnan(gini_train) and not np.isnan(gini_valid):
      ar_diff = abs(gini_train - gini_valid)
      if ar_diff > ar_diff_max:
        reasons.append(f"AR-diff > {ar_diff_max}")

    psi = np.nan
    if grp_col in train_woe.columns and grp_col in valid_woe.columns:
      psi = compute_psi(train_woe[grp_col], valid_woe[grp_col], epsilon)
      if psi > psi_max:
        reasons.append(f"PSI > {psi_max}")
    else:
      reasons.append("No GRP column")

    rows.append(
      {
        "feature": feature,
        "iv": iv,
        "gini_train": gini_train,
        "gini_valid": gini_valid,
        "ar_diff": ar_diff,
        "psi": psi,
        "status": "keep" if not reasons else "reject",
        "reason": "; ".join(reasons),
      }
    )

  return pd.DataFrame(rows)


def assess_logit_model(model, train_df, valid_df, target):
  feature_cols = [col for col in model.params.index if col != "const"]

  x_train = sm.add_constant(train_df[feature_cols], has_constant="add")
  x_valid = sm.add_constant(valid_df[feature_cols], has_constant="add")

  pred_train = model.predict(x_train)
  pred_valid = model.predict(x_valid)

  gini_train = compute_gini(train_df[target], pred_train)
  gini_valid = compute_gini(valid_df[target], pred_valid)
  ar_diff = abs(gini_train - gini_valid)

  pvalues = model.pvalues.drop(labels=["const"], errors="ignore").to_dict()
  max_pvalue = max(pvalues.values()) if pvalues else 0.0

  vif_series = check_vif(train_df[feature_cols])
  vif_dict = vif_series.to_dict()
  max_vif = max(vif_dict.values()) if vif_dict else 1.0

  pearson_corr = train_df[feature_cols].corr(method="pearson")
  off_diag = pearson_corr.where(~np.eye(len(pearson_corr), dtype=bool))
  max_pearson_offdiag = (
    float(off_diag.abs().max().max()) if len(feature_cols) > 1 else 0.0
  )

  betas = model.params.drop(labels=["const"], errors="ignore")
  beta_signs = {
    feat: ("positive" if b > 0 else "negative" if b < 0 else "zero")
    for feat, b in betas.items()
  }
  n_negative_betas = sum(1 for s in beta_signs.values() if s == "negative")

  return {
    "gini_train": gini_train,
    "gini_valid": gini_valid,
    "ar_diff": ar_diff,
    "pvalues": pvalues,
    "max_pvalue": max_pvalue,
    "vif": vif_dict,
    "max_vif": max_vif,
    "pearson_corr": pearson_corr,
    "max_pearson_offdiag": max_pearson_offdiag,
    "beta_signs": beta_signs,
    "n_negative_betas": n_negative_betas,
    "n_features": len(feature_cols),
  }


def forward_select_logit(train_df, valid_df, features, target, params):
  pvalue_max = params["pvalue_max"]
  vif_max = params["vif_max"]
  pearson_max = params["pearson_max"]
  ar_diff_max = params["ar_diff_model_max"]
  max_features = params["max_features"]
  epsilon = params["epsilon"]

  selected = []
  remaining = list(features)
  best_gini_valid = -np.inf

  while remaining and len(selected) < max_features:
    candidates_results = []

    for feat in remaining:
      trial_features = selected + [feat]
      x_train = sm.add_constant(train_df[trial_features], has_constant="add")
      y_train = train_df[target]

      try:
        model = sm.Logit(y_train, x_train).fit(disp=0)
      except Exception:
        continue

      diagnostics = assess_logit_model(model, train_df, valid_df, target)
      if diagnostics["max_pvalue"] > pvalue_max:
        continue
      if diagnostics["max_vif"] > vif_max:
        continue
      if diagnostics["max_pearson_offdiag"] > pearson_max:
        continue
      if diagnostics["ar_diff"] > ar_diff_max:
        continue

      signs = set(diagnostics["beta_signs"].values())
      if "zero" in signs and len(signs) > 1:
        continue
      if "positive" in signs and "negative" in signs:
        continue

      candidates_results.append(
        (feat, diagnostics["gini_valid"], diagnostics["max_pvalue"])
      )

    if not candidates_results:
      break

    def sort_key(item):
      feat, gini_valid, max_pvalue = item
      original_idx = features.index(feat)
      return (-gini_valid, max_pvalue, original_idx)

    candidates_results.sort(key=sort_key)
    best_feat, best_feat_gini, _ = candidates_results[0]

    if best_feat_gini <= best_gini_valid + epsilon:
      break

    selected.append(best_feat)
    remaining.remove(best_feat)
    best_gini_valid = best_feat_gini

  return selected

In [7]:
# --- Model fit, scaling, calibration ---

def train_pd_model(
  product,
  train_df,
  valid_df,
  params,
  *,
  candidate_woe_features=None,
  woe_maps=None,
):
  target = params["target"]
  train_subset = train_df[
    (train_df["product"] == product) & (train_df["decision"] == "A")
  ].copy()
  valid_subset = valid_df[
    (valid_df["product"] == product) & (valid_df["decision"] == "A")
  ].copy()

  if candidate_woe_features is None:
    candidate_woe_features = [col for col in train_subset.columns if col.endswith("_WOE")]

  selected = forward_select_logit(
    train_subset, valid_subset, candidate_woe_features, target, params
  )
  if not selected and candidate_woe_features:
    selected = [candidate_woe_features[0]]

  x_train = sm.add_constant(train_subset[selected], has_constant="add")
  y_train = train_subset[target]
  final_model = sm.Logit(y_train, x_train).fit(disp=0)
  metrics = assess_logit_model(final_model, train_subset, valid_subset, target)

  woe_tables = {}
  if woe_maps is not None:
    for woe_col in selected:
      raw_feat = woe_col[: -len("_WOE")]
      grp_key = f"{raw_feat}_GRP"
      if grp_key in woe_maps:
        woe_tables[woe_col] = woe_maps[grp_key]

  return {
    "product": product,
    "features": selected,
    "model": final_model,
    "metrics": metrics,
    "train_subset": train_subset,
    "valid_subset": valid_subset,
    "woe_tables": woe_tables,
    "id_col": params.get("id_col", "aid"),
  }


def scale_scorecard(model_package, factor, offset):
  model = model_package["model"]
  features = model_package["features"]
  intercept = model.params.get("const", 0.0)
  base_points = offset - factor * intercept
  rows = []

  for feat in features:
    beta = model.params[feat]
    woe_table = model_package.get("woe_tables", {}).get(feat)

    if woe_table is None:
      rows.append({"feature": feat, "bin": "<ALL>", "woe": np.nan, "points": np.nan})
      continue

    for _, row in woe_table.iterrows():
      rows.append(
        {
          "feature": feat,
          "bin": row["bin"],
          "woe": row["woe"],
          "points": -beta * row["woe"] * factor,
        }
      )

  points_table = pd.DataFrame(rows)
  points_table.attrs["base_points"] = base_points
  points_table.attrs["intercept"] = intercept
  points_table.attrs["factor"] = factor
  points_table.attrs["offset"] = offset
  return points_table


def score_applicants(df_woe, model_package, points_table=None):
  features = model_package["features"]
  id_col = model_package.get("id_col", "aid")

  out = pd.DataFrame({id_col: df_woe[id_col].values})

  if points_table is not None:
    base_points = points_table.attrs.get("base_points", 0.0)
    total = pd.Series(base_points, index=df_woe.index, dtype=float)

    for feat in features:
      raw_feat = feat[: -len("_WOE")]
      grp_col = f"{raw_feat}_GRP"
      feat_points_map = (
        points_table[points_table["feature"] == feat]
        .set_index("bin")["points"]
        .to_dict()
      )
      contrib = df_woe[grp_col].map(feat_points_map).fillna(0.0)
      out[f"{feat}_points"] = contrib.values
      total = total + contrib

    out["score"] = total.values
  else:
    model = model_package["model"]
    x = df_woe[features].copy()
    x.insert(0, "const", 1.0)
    out["score"] = x.values @ model.params[["const"] + features].values

  return out.rename(columns={id_col: "aid"}) if id_col != "aid" else out


def calibrate_pd(scores_df, target="default12"):
  y = scores_df[target].values
  score = scores_df["score"].values

  x = sm.add_constant(score)
  calib_model = sm.Logit(y, x).fit(disp=0)

  if hasattr(calib_model.params, "index"):
    param_index = calib_model.params.index.tolist()
    a = float(calib_model.params["const"]) if "const" in param_index else float(calib_model.params.iloc[0])
    slope_cols = [c for c in param_index if c != "const"]
    b = float(calib_model.params[slope_cols[0]]) if slope_cols else 0.0
  else:
    a = float(calib_model.params[0])
    b = float(calib_model.params[1]) if len(calib_model.params) > 1 else 0.0
  pd_calibrated = calib_model.predict(x)

  auc_before = roc_auc_score(y, score)
  auc_after = roc_auc_score(y, pd_calibrated)

  score_norm = (score - score.min()) / (score.max() - score.min() + 1e-12)
  brier_before = brier_score_loss(y, score_norm)
  brier_after = brier_score_loss(y, pd_calibrated)

  diagnostics = {
    "auc_before": float(auc_before),
    "auc_after": float(auc_after),
    "brier_before": float(brier_before),
    "brier_after": float(brier_after),
    "mean_pd_predicted": float(pd_calibrated.mean()),
    "mean_pd_actual": float(y.mean()),
  }

  return {
    "params": {"a": a, "b": b, "target": target, "score_col": "score"},
    "diagnostics": diagnostics,
  }

In [8]:
# --- Orchestration ---

def run_product_scorecard_pipeline(df_model, product, params):
  cand = get_candidate_features(df_model, product)
  all_features = cand["all"]

  df_train, df_valid = partition_abt(
    df_model,
    params["train_end_period"],
    params["valid_start_period"],
    time_col=params["time_col"],
  )

  train_product = df_train[df_train["product"] == product].copy()
  valid_product = df_valid[df_valid["product"] == product].copy()

  binning_maps = fit_binning_maps(train_product, all_features, params["target"], params)
  train_binned = apply_bins(train_product, binning_maps)
  valid_binned = apply_bins(valid_product, binning_maps)

  grp_cols = [f"{f}_GRP" for f in all_features if f"{f}_GRP" in train_binned.columns]
  woe_maps = build_woe_maps(train_binned, grp_cols, params["target"], params["woe_epsilon"])
  iv_table = build_iv_table(woe_maps)

  train_woe = encode_woe(train_binned, woe_maps)
  valid_woe = encode_woe(valid_binned, woe_maps)

  screen_report = prescreen_features(train_woe, valid_woe, iv_table, params)
  selected = screen_report.loc[screen_report["status"] == "keep", "feature"].tolist()
  candidate_woe = [f"{f}_WOE" for f in selected if f"{f}_WOE" in train_woe.columns]

  model_package = train_pd_model(
    product,
    train_woe,
    valid_woe,
    params,
    candidate_woe_features=candidate_woe,
    woe_maps=woe_maps,
  )
  points_table = scale_scorecard(model_package, params["factor"], params["offset"])

  valid_scored = score_applicants(valid_woe, model_package, points_table)
  valid_scored = valid_scored.merge(
    valid_woe[[params["id_col"], params["target"]]].rename(columns={params["id_col"]: "aid"}),
    on="aid",
    how="left",
  )
  calibration = calibrate_pd(valid_scored, target=params["target"])

  return {
    "product": product,
    "candidates": cand,
    "binning_maps": binning_maps,
    "woe_maps": woe_maps,
    "iv_table": iv_table,
    "screen_report": screen_report,
    "train_woe": train_woe,
    "valid_woe": valid_woe,
    "model_package": model_package,
    "points_table": points_table,
    "valid_scores": valid_scored,
    "calibration": calibration,
  }


def run_full_scorecard_pipeline(abt_path, decisions_path, params, output_dir="../data/06_models"):
  df_model = load_and_prepare_abt(abt_path, decisions_path)
  df_model = df_model[df_model["decision"] == "A"].copy()

  results = {
    product: run_product_scorecard_pipeline(df_model, product, params)
    for product in ("ins", "css")
  }

  out = Path(output_dir)
  out.mkdir(parents=True, exist_ok=True)

  with (out / "woe_maps.pkl").open("wb") as fh:
    pickle.dump({p: results[p]["woe_maps"] for p in results}, fh)

  for product in results:
    with (out / f"pd_{product}.pkl").open("wb") as fh:
      pickle.dump(results[product]["model_package"], fh)

    results[product]["points_table"].to_parquet(
      out / f"points_table_{product}.parquet", index=False
    )
    results[product]["screen_report"].to_parquet(
      out / f"feature_screen_report_{product}.parquet", index=False
    )

  calibration_params = {
    product: results[product]["calibration"]["params"] for product in results
  }
  calibration_diagnostics = {
    product: results[product]["calibration"]["diagnostics"] for product in results
  }

  with (out / "calibration_params.json").open("w", encoding="utf-8") as fh:
    json.dump(calibration_params, fh, indent=2)

  with (out / "calibration_diagnostics.json").open("w", encoding="utf-8") as fh:
    json.dump(calibration_diagnostics, fh, indent=2)

  results["calibration_params"] = calibration_params
  results["calibration_diagnostics"] = calibration_diagnostics
  return results

In [9]:
# Prerequisites checkpoint
assert Path("../data/04_feature/abt_app.parquet").exists()
assert Path("../data/04_feature/decisions.parquet").exists()

abt_app = pd.read_parquet("../data/04_feature/abt_app.parquet")
decisions = pd.read_parquet("../data/04_feature/decisions.parquet")
assert abt_app["aid"].is_unique
print(abt_app.shape, decisions.shape)

(49224, 208) (49224, 8)


In [ ]:
# Checkpoint 3: candidate features (accepted-only, no IDs/leakage)
cand_css = get_candidate_features(df, "css")
assert set(cand_css["numeric"]).isdisjoint(set(cand_css["nominal"]))
assert not set(ID_COLS + LEAKAGE_COLS) & set(cand_css["all"])
print("css candidates:", len(cand_css["all"]), "numeric", len(cand_css["numeric"]), "nominal", len(cand_css["nominal"]))


In [ ]:
# Checkpoint 4: partition
df_accepted = df[df["decision"] == "A"].copy()
df_train, df_valid = partition_abt(
    df_accepted,
    SCORECARD_PARAMS["train_end_period"],
    SCORECARD_PARAMS["valid_start_period"],
    time_col=SCORECARD_PARAMS["time_col"],
)
assert df_train["period"].max() == SCORECARD_PARAMS["train_end_period"]
assert df_valid["period"].min() == SCORECARD_PARAMS["valid_start_period"]
print(df_train.shape, df_valid.shape)


In [ ]:
# Checkpoint 5–7: binning (css train slice smoke test)
train_css = df_train[df_train["product"] == "css"]
sample_features = cand_css["all"][:3]
binning_maps = fit_binning_maps(train_css, sample_features, SCORECARD_PARAMS["target"], SCORECARD_PARAMS)

bin_num = fit_bin_numeric(train_css, "app_income", SCORECARD_PARAMS["target"], SCORECARD_PARAMS)
assert bin_num["type"] == "numeric"
assert bin_num["edges"][0] == -np.inf and bin_num["edges"][-1] == np.inf

bin_nom = fit_bin_nominal(train_css, "app_char_job_code", SCORECARD_PARAMS["target"], SCORECARD_PARAMS)
assert bin_nom["type"] == "nominal"
print("sample bin maps:", list(binning_maps))


In [ ]:
# Checkpoint 8: apply bins
train_binned = apply_bins(train_css, binning_maps)
valid_binned = apply_bins(df_valid[df_valid["product"] == "css"], binning_maps)
grp_cols = [c for c in train_binned.columns if c.endswith("_GRP")]
assert len(grp_cols) == len(binning_maps)
assert train_binned.columns.duplicated().sum() == 0
display(train_binned[grp_cols].head())


In [ ]:
# Checkpoint 9–11: WOE + IV
woe_table = build_woe_table(train_binned, "act_age_GRP", SCORECARD_PARAMS["target"], SCORECARD_PARAMS["woe_epsilon"])
assert {"bin", "woe", "iv_component"} <= set(woe_table.columns)
iv_value = compute_iv(woe_table)
assert iv_value >= 0
print(f"act_age IV: {iv_value:.4f}")


In [ ]:
# Checkpoint 10: encode WOE (full css product)
all_features = cand_css["all"]
binning_maps_full = fit_binning_maps(train_css, all_features, SCORECARD_PARAMS["target"], SCORECARD_PARAMS)
train_binned_full = apply_bins(train_css, binning_maps_full)
valid_binned_full = apply_bins(df_valid[df_valid["product"] == "css"], binning_maps_full)
grp_cols_full = [f"{f}_GRP" for f in all_features if f"{f}_GRP" in train_binned_full.columns]
woe_maps = build_woe_maps(train_binned_full, grp_cols_full, SCORECARD_PARAMS["target"], SCORECARD_PARAMS["woe_epsilon"])
iv_table = build_iv_table(woe_maps)
train_woe = encode_woe(train_binned_full, woe_maps)
valid_woe = encode_woe(valid_binned_full, woe_maps)
assert train_woe.columns.duplicated().sum() == 0
print("WOE cols:", len([c for c in train_woe.columns if c.endswith('_WOE')]))


In [ ]:
# Checkpoint 12–14: metrics + prescreen
gini = compute_gini(train_woe[SCORECARD_PARAMS["target"]], train_woe["app_income_WOE"])
psi = compute_psi(train_woe["act_age_GRP"], valid_woe["act_age_GRP"], SCORECARD_PARAMS["woe_epsilon"])
assert 0 <= gini <= 1
assert psi >= 0

screen_report = prescreen_features(train_woe, valid_woe, iv_table, SCORECARD_PARAMS)
assert set(screen_report["status"]) <= {"keep", "reject"}
print("kept:", (screen_report["status"] == "keep").sum(), "of", len(screen_report))
display(screen_report.head(10))


In [ ]:
# Checkpoint 15: VIF on WOE design matrix
woe_cols = [c for c in train_woe.columns if c.endswith("_WOE")]
vif_table = check_vif(train_woe[woe_cols[:10]])
assert (vif_table.dropna() >= 1.0).all()
display(vif_table.head())


In [ ]:
# Checkpoint 16–18: train css model on prescreen survivors
selected = screen_report.loc[screen_report["status"] == "keep", "feature"].tolist()
candidate_woe = [f"{f}_WOE" for f in selected if f"{f}_WOE" in train_woe.columns]
pd_css = train_pd_model(
    "css",
    train_woe,
    valid_woe,
    SCORECARD_PARAMS,
    candidate_woe_features=candidate_woe,
    woe_maps=woe_maps,
)
assert len(pd_css["features"]) >= 1
print("selected features:", len(pd_css["features"]))
print("valid gini:", f"{pd_css['metrics']['gini_valid']:.2%}")


In [ ]:
# Checkpoint 19–21: scale, score, calibrate (css)
points_table = scale_scorecard(pd_css, SCORECARD_PARAMS["factor"], SCORECARD_PARAMS["offset"])
assert {"feature", "bin", "points"} <= set(points_table.columns)
assert np.isfinite(points_table.attrs["base_points"])

scores = score_applicants(valid_woe, pd_css, points_table)
assert {"aid", "score"} <= set(scores.columns)
assert scores["aid"].is_unique

scores_cal = scores.merge(valid_woe[["aid", SCORECARD_PARAMS["target"]]], on="aid")
calibration = calibrate_pd(scores_cal, target=SCORECARD_PARAMS["target"])
assert "params" in calibration and "diagnostics" in calibration
print(calibration["params"])


In [ ]:
# Full pipeline (ins + css) — expect ~2–4 min
results = run_full_scorecard_pipeline(
    "../data/04_feature/abt_app.parquet",
    "../data/04_feature/decisions.parquet",
    SCORECARD_PARAMS,
    output_dir="../data/06_models",
)

# --- Parity checks (phase_4_scorecard.md) ---
for product in ("ins", "css"):
    gini_valid = results[product]["model_package"]["metrics"]["gini_valid"]
    n_features = len(results[product]["model_package"]["features"])
    print(f"PD {product}: {n_features} features, valid Gini {gini_valid:.1%}")

assert results["ins"]["model_package"]["metrics"]["gini_valid"] > 0.30
assert results["css"]["model_package"]["metrics"]["gini_valid"] > 0.30

print("Saved woe_maps.pkl, pd_ins.pkl, pd_css.pkl, points tables, calibration JSON")
